# PRIK quickstart

Compile Fortran and C in notebook cells and call them from Python.

This notebook runs top to bottom. Every result is checked against what
the text says it should be, so a ✅ means the cell really did that.

## Setup

Colab needs a Fortran compiler and PRIK itself.

In [ ]:
!apt-get -qq install -y gfortran > /dev/null
!pip install -q "prik[jupyter] @ git+https://github.com/PyNumLab/prik.git"

If pip upgraded NumPy, Colab will ask you to restart the runtime. Do that,
then continue from the next cell.

In [ ]:
%load_ext prik.jupyter

In [ ]:
import numpy as np

## 1. Compile a Fortran cell

`%%fortran` compiles the cell and publishes what it declares. A Fortran
module becomes a notebook name.

In [ ]:
%%fortran
module geometry
contains
    real(8) function circle_area(radius)
        real(8), intent(in) :: radius
        circle_area = 3.141592653589793d0 * radius**2
    end function
end module


In [ ]:
area = geometry.circle_area(np.float64(2.0))
assert np.isclose(area, np.pi * 4)
print(f"✅ circle_area(2.0) = {area}  (expected {np.pi * 4})")

## 2. Compile a C cell

`%%c` publishes C functions directly. This one doubles an array in place,
and takes the element count the way C usually does.

In [ ]:
%%c
#include <stddef.h>

void scale(size_t count, double *values) {
    for (size_t index = 0; index < count; ++index) {
        values[index] *= 2.0;
    }
}


PRIK gave `double *values` a *runtime-rank* contract, so it accepts a NumPy
array of any rank and writes through it. The count still has to be passed
by hand, even though NumPy already knows it:

In [ ]:
values = np.array([1.0, 2.0, 3.0])
scale(np.uintp(values.size), values)
assert np.allclose(values, [2.0, 4.0, 6.0])
print(f"✅ scale(count, values) doubled in place: {values}  (expected [2. 4. 6.])")

## 3. Reshape the API with a contract

`--pyi` compiles nothing. It keeps the source and hands back the semantic
contract it derived, as an editable cell.

In Jupyter and Colab the cell appears below this one automatically. It is
already written out for you here so the notebook runs end to end.

In [ ]:
%%c --pyi
#include <stddef.h>

void scale(size_t count, double *values) {
    for (size_t index = 0; index < count; ++index) {
        values[index] *= 2.0;
    }
}


The generated contract reads `def scale(count: UInt64, values: Float64[...])`.
Edit it so `count` comes from the array instead: `Arg(0).size` supplies it,
and `Float64[:]` pins the rank to one.

In [ ]:
%%pyi

# prik: source-sha256=cbb931db2a66b26b4c8f1796b7fe1f93ec5c9af2e76fdacfb91c191a14055def

from prik.contracts import Arg, Float64, native_call

@native_call([Arg(0).size, Arg(0)])
def scale(values: Float64[:]) -> None: ...


Same C code, same compiler. Only the Python API changed — `count` is gone:

In [ ]:
values = np.array([1.0, 2.0, 3.0])
scale(values)
assert np.allclose(values, [2.0, 4.0, 6.0])
print(f"✅ scale(values) doubled in place: {values}  (expected [2. 4. 6.])")

In [ ]:
print("🎉 All checks passed.")
print("   Fortran and C both compiled in this notebook,")
print("   and the C API was reshaped by editing its contract.")

## Where to go next

- [IPython and Jupyter Notebooks](https://pynumlab.github.io/prik/user/guide/notebooks/)
  — every magic, option, and the cell cache
- [C Pointers, Arrays, and Strings](https://pynumlab.github.io/prik/user/guide/c/pointers-arrays-and-strings/)
  — what `Float64[...]` accepts and how to narrow it
- [Design a Pythonic BLAS API](https://pynumlab.github.io/prik/user/tutorials/pythonic-blas/)
  — the same contract editing, applied to a real library